In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc
from sqlalchemy import create_engine
from urllib.parse import quote_plus

SQL_SERVER_HOST = "localhost"
SQL_SERVER_DATABASE = "typing_test"
SQL_SERVER_USERNAME = "sa"
SQL_SERVER_PASSWORD = "Toughpass1!"
ODBC_DRIVER = "ODBC Driver 18 for SQL Server"

# Set DRY_RUN = True to validate connection + query shape with only a small sample.
DRY_RUN = True
DRY_RUN_LIMIT = 25

CONNECTION_STRING = (
    f"mssql+pyodbc://{SQL_SERVER_USERNAME}:{quote_plus(SQL_SERVER_PASSWORD)}@"
    f"{SQL_SERVER_HOST}/{SQL_SERVER_DATABASE}"
    f"?driver={quote_plus(ODBC_DRIVER)}"
    "&TrustServerCertificate=yes"
)


def build_training_query(dry_run=False, row_limit=25):
    top_clause = f"TOP ({int(row_limit)}) " if dry_run and row_limit > 0 else ""

    return f"""
WITH PendingTrainingRows AS (
    SELECT
        sf.Id AS SessionFeatureId,
        ts.Id AS TypingSessionId,
        sf.CreatedAt AS FeatureCapturedAt,
        ts.CreatedAt AS SessionCapturedAt,
        CAST(ts.Wpm AS float) AS Wpm,
        CAST(CASE WHEN ts.Accuracy <= 1 THEN ts.Accuracy * 100.0 ELSE ts.Accuracy END AS float) AS Accuracy,
        CAST(sf.DwellLeftPinky AS float) AS dwell_left_pinky,
        CAST(sf.DwellLeftRing AS float) AS dwell_left_ring,
        CAST(sf.DwellLeftMiddle AS float) AS dwell_left_middle,
        CAST(sf.DwellLeftIndex AS float) AS dwell_left_index,
        CAST(sf.DwellRightIndex AS float) AS dwell_right_index,
        CAST(sf.DwellRightMiddle AS float) AS dwell_right_middle,
        CAST(sf.DwellRightRing AS float) AS dwell_right_ring,
        CAST(sf.DwellRightPinky AS float) AS dwell_right_pinky,
        CAST(sf.FlightLeftPinky AS float) AS flight_left_pinky,
        CAST(sf.FlightLeftRing AS float) AS flight_left_ring,
        CAST(sf.FlightLeftMiddle AS float) AS flight_left_middle,
        CAST(sf.FlightLeftIndex AS float) AS flight_left_index,
        CAST(sf.FlightRightIndex AS float) AS flight_right_index,
        CAST(sf.FlightRightMiddle AS float) AS flight_right_middle,
        CAST(sf.FlightRightRing AS float) AS flight_right_ring,
        CAST(sf.FlightRightPinky AS float) AS flight_right_pinky,
        CASE
            WHEN LOWER(LTRIM(RTRIM(ISNULL(ts.WeakestFinger, '')))) IN (
                'left_pinky', 'left_ring', 'left_middle', 'left_index',
                'right_index', 'right_middle', 'right_ring', 'right_pinky'
            )
                THEN LOWER(LTRIM(RTRIM(ts.WeakestFinger)))
            ELSE inferred.finger_name
        END AS weakest_finger
    FROM SessionFeature sf
    INNER JOIN TypingSession ts
        ON ts.Id = sf.TypingSessionId
    OUTER APPLY (
        SELECT TOP (1) errors.finger_name
        FROM (VALUES
            ('left_pinky', CAST(sf.ErrorLeftPinky AS float)),
            ('left_ring', CAST(sf.ErrorLeftRing AS float)),
            ('left_middle', CAST(sf.ErrorLeftMiddle AS float)),
            ('left_index', CAST(sf.ErrorLeftIndex AS float)),
            ('right_index', CAST(sf.ErrorRightIndex AS float)),
            ('right_middle', CAST(sf.ErrorRightMiddle AS float)),
            ('right_ring', CAST(sf.ErrorRightRing AS float)),
            ('right_pinky', CAST(sf.ErrorRightPinky AS float))
        ) AS errors(finger_name, error_value)
        ORDER BY errors.error_value DESC, errors.finger_name ASC
    ) AS inferred
    WHERE sf.IsUsedForTraining = 0
)
SELECT {top_clause}
    ROW_NUMBER() OVER (ORDER BY FeatureCapturedAt ASC) AS Sesi,
    TypingSessionId,
    SessionFeatureId,
    SessionCapturedAt,
    FeatureCapturedAt,
    weakest_finger,
    CAST((
        flight_left_pinky + flight_left_ring + flight_left_middle + flight_left_index +
        flight_right_index + flight_right_middle + flight_right_ring + flight_right_pinky
    ) / 8.0 AS float) AS FlightTime,
    CAST((
        dwell_left_pinky + dwell_left_ring + dwell_left_middle + dwell_left_index +
        dwell_right_index + dwell_right_middle + dwell_right_ring + dwell_right_pinky
    ) / 8.0 AS float) AS DwellTime
FROM PendingTrainingRows
WHERE weakest_finger IS NOT NULL
ORDER BY Sesi;
"""


TRAINING_QUERY = build_training_query(DRY_RUN, DRY_RUN_LIMIT)

try:
    mode_label = f"DRY RUN ({DRY_RUN_LIMIT} rows max)" if DRY_RUN else "FULL RUN"
    print(f"1. Mode eksekusi: {mode_label}")
    print("2. Mencoba terhubung ke SQL Server...")
    engine = create_engine(CONNECTION_STRING)

    print("3. Menjalankan query training source...")
    df_source = pd.read_sql(TRAINING_QUERY, engine)

    print(f"4. Berhasil! Ditemukan {len(df_source)} baris data dari query.")

    if len(df_source) == 0:
        print("⚠️ PERINGATAN: Koneksi sukses, tetapi query tidak mengembalikan data. Pastikan ada SessionFeature yang belum dipakai untuk training.")
        df_raw = pd.DataFrame(columns=["Sesi", "FlightTime", "DwellTime"])
    else:
        df_raw = df_source[["Sesi", "FlightTime", "DwellTime"]].copy()
        print("\n--- PREVIEW DATA (5 BARIS PERTAMA) ---")
        print(df_raw.head())

except Exception as e:
    print("\n❌ TERJADI ERROR SAAT KONEKSI / QUERY:")
    print(e)
    df_source = pd.DataFrame()
    df_raw = pd.DataFrame(columns=["Sesi", "FlightTime", "DwellTime"])

In [ ]:
def remove_outliers_iqr(df, column_name):
    numeric_series = pd.to_numeric(df[column_name], errors="coerce")
    q1 = numeric_series.quantile(0.25)
    q3 = numeric_series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    filtered_df = df.loc[
        numeric_series.notna()
        & (numeric_series >= lower_bound)
        & (numeric_series <= upper_bound)
    ].copy()
    return filtered_df.reset_index(drop=True)

df_raw = df_raw.copy()
df_clean = remove_outliers_iqr(df_raw, "DwellTime")

print(f"Jumlah data sumber query training: {len(df_source)}")
print(f"Jumlah data sebelum IQR: {len(df_raw)}")
print(f"Jumlah data sesudah IQR: {len(df_clean)}")
print(f"Jumlah outlier yang dihapus: {len(df_raw) - len(df_clean)}")

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

if df_raw.empty or df_clean.empty:
    print("Tidak ada data yang cukup untuk divisualisasikan. Jalankan Cell 1 dalam mode dry run/full run dan pastikan query mengembalikan data.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    fig.suptitle("Perbandingan Distribusi Waktu Dwell Sebelum dan Sesudah IQR", fontsize=16, fontweight="bold")

    sns.boxplot(data=df_raw, y="DwellTime", ax=axes[0], color="#d95f02", width=0.35)
    axes[0].set_title("Waktu Dwell (ms) Sebelum Pembersihan", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Data Mentah")
    axes[0].set_ylabel("Waktu Dwell (ms)")

    sns.boxplot(data=df_clean, y="DwellTime", ax=axes[1], color="#1b9e77", width=0.35)
    axes[1].set_title("Waktu Dwell (ms) Sesudah Pembersihan IQR", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Data Bersih")
    axes[1].set_ylabel("Waktu Dwell (ms)")

    for ax in axes:
        ax.grid(True, axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()